# Полный цикл alignment: SFT, style-DPO, Reward Model, quality-DPO и SimPO

Один полностью воспроизводимый Colab-ноутбук для всех пяти задач. Модель и все адаптеры обучаются
в этой сессии с нуля от зафиксированной ревизии `Qwen/Qwen3-4B-Instruct-2507`; готовые адаптеры
извне не загружаются. Публичные наборы используются только для финального измерения соответствующего
этапа, никогда не передаются тренерам.

Перед запуском: **Runtime → Change runtime type → T4 GPU**, затем **Runtime → Run all**.


## 0. Зафиксированное окружение

NumPy, PyTorch и CUDA оставляем штатными для текущего Colab-образа, чтобы не разрушить бинарную
совместимость. Остальные зависимости фиксируются точными версиями.


In [ ]:
import subprocess
import sys

PINNED_PACKAGES = [
    "transformers==4.57.1",
    "trl==0.24.0",
    "peft==0.17.1",
    "accelerate==1.10.1",
    "bitsandbytes==0.48.1",
    "datasets==4.1.1",
    "scikit-learn==1.7.2",
    "scipy==1.16.2",
    "pandas==2.3.3",
    "joblib==1.5.2",
    "sentencepiece==0.2.1",
    "tokenizers==0.22.1",
    "safetensors==0.6.2",
    "huggingface-hub==0.35.3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", *PINNED_PACKAGES]
)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], text=True, capture_output=True
)
print("pip check return code:", pip_check.returncode)
print((pip_check.stdout + pip_check.stderr).strip() or "No dependency conflicts reported.")
print("Pinned task packages installed; Colab NumPy/PyTorch/CUDA were left untouched.")


In [ ]:
import importlib.metadata as metadata
import platform

import torch

distributions = [
    "numpy", "transformers", "trl", "peft", "accelerate", "bitsandbytes",
    "datasets", "scikit-learn", "scipy", "pandas", "joblib", "sentencepiece",
    "tokenizers", "safetensors", "huggingface-hub",
]
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
for distribution in distributions:
    print(f"{distribution}: {metadata.version(distribution)}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. Select Runtime -> Change runtime type -> T4 GPU and run all again."
    )
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {props.total_memory / 2**30:.2f} GiB")
subprocess.run(["nvidia-smi"], check=True)


## 1. Конфигурация, seed и входные данные

Все параметры находятся в одной ячейке. Хэши защищают от незаметной замены датасетов или измерителя.
`good_bad` и `kid_adult` загружаются до обучения; публичные наборы будут открыты только в ячейках оценки.


In [ ]:
import gc
import hashlib
import json
import os
import random
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import set_seed

CONFIG = {
    "repo_url": "https://github.com/george1r/-utask.git",
    "repo_branch": "main",
    "model_id": "Qwen/Qwen3-4B-Instruct-2507",
    "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "kid_adult_sha256": "52bacff1c6d5d50ca3dd473f8d754cf1dfcce7e02ecf162cda2c18719a138748",
    "good_bad_sha256": "bf50f3af0127df71d891c5a65eb75220104157f3e27b613aacbae1761c08998b",
    "public_test_style_sha256": "d0f5fb848245b18e97b97fe5158c602f3f2b49b8ec6588f93a0f0e9f10c58efe",
    "public_test_quality_sha256": "bc8b21bf04c88e99d420569c61f46309f71d04601159b80ea258760e8d871780",
    "style_clf_sha256": "b5cf7b53417033de19b9c44a43402bce0e6eeece44b1abac2cf596785b60888d",
    "seed": 42,
    "sft_epochs": 2,
    "sft_lr": 1e-4,
    "style_dpo_epochs": 1,
    "style_dpo_lr": 1e-5,
    "quality_rm_epochs": 2,
    "quality_rm_lr": 5e-5,
    "quality_dpo_epochs": 1,
    "quality_dpo_lr": 1e-5,
    "simpo_epochs": 1,
    "simpo_lr": 5e-6,
    "dpo_beta": 0.1,
    "simpo_beta": 2.0,
    "simpo_gamma": 1.0,
    "simpo_cpo_alpha": 0.0,
    "train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "sft_max_length": 512,
    "style_dpo_max_prompt_length": 256,
    "style_dpo_max_length": 512,
    "quality_max_prompt_length": 128,
    "quality_max_length": 384,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "optim": "paged_adamw_8bit",
    "max_grad_norm": 0.3,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "generation_max_new_tokens": 192,
}
SEED = CONFIG["seed"]
assert SEED == 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

def reset_seeds() -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

reset_seeds()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = False
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("Effective training batch:", CONFIG["train_batch_size"] * CONFIG["gradient_accumulation_steps"])


In [ ]:
repo_candidates = [Path.cwd(), Path("/content/-utask")]
repo_root = next(
    (p for p in repo_candidates if (p / "data" / "kid_adult.jsonl").is_file()),
    None,
)
if repo_root is None:
    repo_root = Path("/content/-utask")
    if repo_root.exists():
        shutil.rmtree(repo_root)
    subprocess.run(
        ["git", "clone", "--branch", CONFIG["repo_branch"], "--single-branch",
         CONFIG["repo_url"], str(repo_root)],
        check=True,
    )
required = [
    "data/kid_adult.jsonl", "data/good_bad.jsonl",
    "data/public_test_style.jsonl", "data/public_test_quality.jsonl",
    "metrics/style_clf.pkl",
]
for relative in required:
    assert (repo_root / relative).is_file(), f"Missing repository file: {relative}"
commit = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", repo_root)
print("Git commit:", commit)


In [ ]:
def read_jsonl_with_sha256(path: Path) -> tuple[list[dict], str]:
    payload = path.read_bytes()
    rows = [json.loads(line) for line in payload.decode("utf-8").splitlines() if line.strip()]
    return rows, hashlib.sha256(payload).hexdigest()

kid_path = repo_root / "data" / "kid_adult.jsonl"
quality_path = repo_root / "data" / "good_bad.jsonl"
style_public_path = repo_root / "data" / "public_test_style.jsonl"
quality_public_path = repo_root / "data" / "public_test_quality.jsonl"
style_clf_path = repo_root / "metrics" / "style_clf.pkl"

kid_rows, kid_hash = read_jsonl_with_sha256(kid_path)
quality_rows, quality_hash = read_jsonl_with_sha256(quality_path)
assert kid_hash == CONFIG["kid_adult_sha256"] and len(kid_rows) == 1489
assert quality_hash == CONFIG["good_bad_sha256"] and len(quality_rows) == 2226
assert all(set(row) == {"prompt", "kid", "adult"} for row in kid_rows)
assert all(set(row) == {"instruction", "chosen", "rejected"} for row in quality_rows)
assert all(all(isinstance(v, str) and v.strip() for v in row.values()) for row in kid_rows + quality_rows)

sft_dataset = Dataset.from_list([
    {"prompt": [{"role": "user", "content": row["prompt"]}],
     "completion": [{"role": "assistant", "content": row["kid"]}]}
    for row in kid_rows
])
style_dpo_dataset = Dataset.from_list([
    {"prompt": [{"role": "user", "content": row["prompt"]}],
     "chosen": [{"role": "assistant", "content": row["kid"]}],
     "rejected": [{"role": "assistant", "content": row["adult"]}]}
    for row in kid_rows
])
quality_pref_dataset = Dataset.from_list([
    {"prompt": [{"role": "user", "content": row["instruction"]}],
     "chosen": [{"role": "assistant", "content": row["chosen"]}],
     "rejected": [{"role": "assistant", "content": row["rejected"]}]}
    for row in quality_rows
])
quality_rm_dataset = Dataset.from_list([
    {"chosen": [
         {"role": "user", "content": row["instruction"]},
         {"role": "assistant", "content": row["chosen"]},
     ],
     "rejected": [
         {"role": "user", "content": row["instruction"]},
         {"role": "assistant", "content": row["rejected"]},
     ]}
    for row in quality_rows
])
assert len(sft_dataset) == len(style_dpo_dataset) == 1489
assert len(quality_pref_dataset) == len(quality_rm_dataset) == 2226
print("Training rows:", {"SFT/style-DPO": 1489, "RM/quality preference": 2226})
print("Public files have not been read.")


## 2. Общие функции модели и измерителей

QLoRA: NF4, double quantization, FP16 compute. Для implicit-preference метрики считается средний
logprob **только токенов ответа**: chat-префикс и служебный конец сообщения не входят в сумму.


In [ ]:
import joblib
from IPython.display import display
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from scipy.sparse import hstack
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_id"], revision=CONFIG["model_revision"], use_fast=True
)
assert tokenizer.chat_template
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
runtime_root = Path("/content") if Path("/content").is_dir() else Path(tempfile.gettempdir())

def causal_lora_config() -> LoraConfig:
    return LoraConfig(
        r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=CONFIG["lora_dropout"], bias="none", task_type="CAUSAL_LM",
        target_modules=CONFIG["lora_target_modules"],
    )

def load_quantized_causal_base():
    base = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"], revision=CONFIG["model_revision"],
        quantization_config=quantization_config, device_map={"": 0},
        torch_dtype=torch.float16, low_cpu_mem_usage=True,
    )
    assert getattr(base, "is_loaded_in_4bit", False)
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(
        base, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return base

def load_style_checkpoint_trainable(adapter_path: Path):
    reset_seeds()
    base = load_quantized_causal_base()
    loaded = PeftModel.from_pretrained(
        base, adapter_path, adapter_name="default", is_trainable=True
    )
    loaded.set_adapter("default")
    return loaded

def release_trainer(trainer) -> None:
    if trainer is not None:
        trainer.optimizer = None
        trainer.lr_scheduler = None
    gc.collect()
    torch.cuda.empty_cache()

def generate_greedy(active_model, prompts: list[str]) -> list[str]:
    reset_seeds()
    active_model.gradient_checkpointing_disable()
    active_model.config.use_cache = True
    active_model.eval()
    answers = []
    for index, prompt in enumerate(prompts):
        input_ids = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=True,
            add_generation_prompt=True, return_tensors="pt",
        ).to(active_model.device)
        with torch.inference_mode():
            output_ids = active_model.generate(
                input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
                do_sample=False, num_beams=1,
                max_new_tokens=CONFIG["generation_max_new_tokens"],
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id, use_cache=True,
            )
        answer = tokenizer.decode(
            output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
        ).strip()
        assert answer
        answers.append(answer)
        print(f"Generated {index + 1:02d}/{len(prompts)}")
    active_model.config.use_cache = False
    active_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return answers

style_hash = hashlib.sha256(style_clf_path.read_bytes()).hexdigest()
assert style_hash == CONFIG["style_clf_sha256"]
style_bundle = joblib.load(style_clf_path)
style_clf = style_bundle["clf"]
word_tfidf, char_tfidf = style_bundle["vecs"]
simple_indices = np.flatnonzero(np.asarray(style_clf.classes_) == 1)
assert len(simple_indices) == 1
simple_index = int(simple_indices[0])

quality_train_full_lengths = []
quality_train_prompt_lengths = []
for row in quality_rows:
    prompt_messages = [{"role": "user", "content": row["instruction"]}]
    quality_train_prompt_lengths.append(len(tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    )))
    for answer_field in ("chosen", "rejected"):
        quality_train_full_lengths.append(len(tokenizer.apply_chat_template(
            prompt_messages + [{"role": "assistant", "content": row[answer_field]}],
            tokenize=True, add_generation_prompt=False,
        )))
assert max(quality_train_prompt_lengths) <= CONFIG["quality_max_prompt_length"]
assert max(quality_train_full_lengths) <= CONFIG["quality_max_length"]
print("Quality train max prompt/full tokens:",
      max(quality_train_prompt_lengths), max(quality_train_full_lengths))

def p_simple(texts: list[str]) -> np.ndarray:
    features = hstack(
        [word_tfidf.transform(texts), char_tfidf.transform(texts)], format="csr"
    )
    assert features.shape == (len(texts), style_clf.n_features_in_)
    values = style_clf.predict_proba(features)[:, simple_index]
    assert np.all(np.isfinite(values)) and np.all((0 <= values) & (values <= 1))
    return values

def interval(value: float, specs: list[tuple[str, str, object]]) -> tuple[str, str]:
    matches = [(letter, bounds) for letter, bounds, predicate in specs if predicate(value)]
    assert len(matches) == 1, matches
    return matches[0]


## 3. Задача 1 — SFT на `kid_adult`

В loss входят только токены простого ответа (`completion_only_loss=True`). После обучения немедленно
измеряется Task 1 на 50 публичных prompts, greedy, seed 42.


In [ ]:
from trl import SFTConfig, SFTTrainer

reset_seeds()
model = get_peft_model(load_quantized_causal_base(), causal_lora_config())
model.print_trainable_parameters()
sft_adapter_dir = runtime_root / "alignment_sft_adapter"
sft_args = SFTConfig(
    output_dir=str(runtime_root / "alignment_sft_training"),
    num_train_epochs=CONFIG["sft_epochs"], learning_rate=CONFIG["sft_lr"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_length=CONFIG["sft_max_length"], fp16=True, bf16=False,
    optim=CONFIG["optim"], warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"], max_grad_norm=CONFIG["max_grad_norm"],
    logging_steps=10, save_strategy="no", report_to="none",
    packing=False, completion_only_loss=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED, data_seed=SEED, dataset_num_proc=1,
)
sft_trainer = SFTTrainer(
    model=model, args=sft_args, train_dataset=sft_dataset, processing_class=tokenizer
)
sft_result = sft_trainer.train()
assert sft_result.global_step > 0
model.save_pretrained(sft_adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(sft_adapter_dir)
print("SFT metrics:", sft_result.metrics)
print("Runtime SFT adapter:", sft_adapter_dir)


In [ ]:
style_public_rows, style_public_hash = read_jsonl_with_sha256(style_public_path)
assert style_public_hash == CONFIG["public_test_style_sha256"]
assert len(style_public_rows) == 50
assert all({"prompt", "kid", "adult"}.issubset(row) for row in style_public_rows)
assert set(row["prompt"] for row in style_public_rows).isdisjoint(
    row["prompt"] for row in kid_rows
)
print("Task 1 public rows loaded after SFT:", len(style_public_rows))

task1_answers = generate_greedy(model, [row["prompt"] for row in style_public_rows])
task1_probs = p_simple(task1_answers)
task1_metric = float(np.mean(task1_probs))
task1_letter, task1_bounds = interval(task1_metric, [
    ("А", "0.4 <= P_simple < 0.7", lambda x: 0.4 <= x < 0.7),
    ("Б", "P_simple < 0.4", lambda x: x < 0.4),
    ("В", "0.7 <= P_simple < 0.9", lambda x: 0.7 <= x < 0.9),
    ("Г", "0.9 <= P_simple <= 1.0", lambda x: 0.9 <= x <= 1.0),
])
print("TASK1_NUM_GENERATIONS=50")
print(f"TASK1_SEED={SEED}")
print("TASK1_DO_SAMPLE=False")
print(f"TASK1_MEAN_P_SIMPLE={task1_metric:.6f}")
print(f"TASK1_INTERVAL={task1_bounds}")
print(f"TASK1_ANSWER={task1_letter}")


## 4. Задача 2 — DPO по стилю

SFT-адаптер копируется внутри текущего runtime: `default` — обучаемая policy, `reference` — замороженная
копия с теми же начальными весами. `chosen=kid`, `rejected=adult`.


In [ ]:
release_trainer(sft_trainer)
del sft_trainer
model.load_adapter(sft_adapter_dir, adapter_name="reference", is_trainable=False)
model.set_adapter("default")

def normalized_adapter_parameters(active_model, adapter_name: str):
    marker = f".{adapter_name}."
    return {
        name.replace(marker, ".<adapter>."): parameter
        for name, parameter in active_model.named_parameters() if marker in name
    }

policy_params = normalized_adapter_parameters(model, "default")
reference_params = normalized_adapter_parameters(model, "reference")
assert policy_params.keys() == reference_params.keys() and policy_params
assert all(torch.equal(policy_params[k].detach(), reference_params[k].detach()) for k in policy_params)
assert all(p.requires_grad for p in policy_params.values())
assert not any(p.requires_grad for p in reference_params.values())

from trl import DPOConfig, DPOTrainer

style_dpo_args = DPOConfig(
    output_dir=str(runtime_root / "alignment_style_dpo_training"),
    num_train_epochs=CONFIG["style_dpo_epochs"], learning_rate=CONFIG["style_dpo_lr"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_prompt_length=CONFIG["style_dpo_max_prompt_length"],
    max_length=CONFIG["style_dpo_max_length"], beta=CONFIG["dpo_beta"],
    loss_type="sigmoid", model_adapter_name="default", ref_adapter_name="reference",
    fp16=True, bf16=False, optim=CONFIG["optim"], warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"], max_grad_norm=CONFIG["max_grad_norm"],
    logging_steps=10, logging_first_step=True, save_strategy="no", report_to="none",
    disable_dropout=True, precompute_ref_log_probs=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED, data_seed=SEED, dataset_num_proc=1,
)
style_dpo_trainer = DPOTrainer(
    model=model, ref_model=None, args=style_dpo_args,
    train_dataset=style_dpo_dataset, processing_class=tokenizer,
)
style_dpo_result = style_dpo_trainer.train()
assert style_dpo_result.global_step > 0
style_logs = [x for x in style_dpo_trainer.state.log_history if "rewards/margins" in x]
assert style_logs
style_adapter_dir = runtime_root / "alignment_style_dpo_adapter"
model.set_adapter("default")
model.save_pretrained(
    style_adapter_dir, safe_serialization=True, selected_adapters=["default"]
)
tokenizer.save_pretrained(style_adapter_dir)
print("Style-DPO metrics:", style_dpo_result.metrics)
print("Reward margin first/last:", style_logs[0]["rewards/margins"], style_logs[-1]["rewards/margins"])
print("Runtime style-DPO adapter:", style_adapter_dir)


In [ ]:
release_trainer(style_dpo_trainer)
del style_dpo_trainer
model.set_adapter("default")
task2_answers = generate_greedy(model, [row["prompt"] for row in style_public_rows])
task2_probs = p_simple(task2_answers)
task2_metric = float(np.mean(task2_probs))
task2_letter, task2_bounds = interval(task2_metric, [
    ("А", "0.4 <= P_simple < 0.7", lambda x: 0.4 <= x < 0.7),
    ("Б", "P_simple < 0.4", lambda x: x < 0.4),
    ("В", "0.7 <= P_simple < 0.9", lambda x: 0.7 <= x < 0.9),
    ("Г", "0.9 <= P_simple <= 1.0", lambda x: 0.9 <= x <= 1.0),
])
print("TASK2_NUM_GENERATIONS=50")
print(f"TASK2_SEED={SEED}")
print("TASK2_DO_SAMPLE=False")
print(f"TASK2_MEAN_P_SIMPLE={task2_metric:.6f}")
print(f"TASK2_INTERVAL={task2_bounds}")
print(f"TASK2_ANSWER={task2_letter}")


## 5. Задача 3 — Reward Model на `good_bad`

Causal policy полностью выгружается. От той же зафиксированной базовой ревизии создаётся отдельная
4-bit sequence-classification модель с одним scalar score. LoRA и новый `score` head обучаются внутри
ноутбука pairwise loss-ом `-log sigmoid(r_chosen-r_rejected)`.


In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
reset_seeds()

rm_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_id"], revision=CONFIG["model_revision"], num_labels=1,
    quantization_config=quantization_config, device_map={"": 0},
    torch_dtype=torch.float16, low_cpu_mem_usage=True,
)
assert getattr(rm_model, "is_loaded_in_4bit", False)
rm_model.config.pad_token_id = tokenizer.pad_token_id
rm_model.config.use_cache = False
rm_model = prepare_model_for_kbit_training(
    rm_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
rm_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
rm_lora = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"], bias="none", task_type="SEQ_CLS",
    target_modules=CONFIG["lora_target_modules"], modules_to_save=["score"],
)
rm_model = get_peft_model(rm_model, rm_lora)
rm_model.print_trainable_parameters()
assert any("score" in name and p.requires_grad for name, p in rm_model.named_parameters())


In [ ]:
from trl import RewardConfig, RewardTrainer

rm_args = RewardConfig(
    output_dir=str(runtime_root / "alignment_reward_model_training"),
    num_train_epochs=CONFIG["quality_rm_epochs"], learning_rate=CONFIG["quality_rm_lr"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_length=CONFIG["quality_max_length"], center_rewards_coefficient=0.01,
    fp16=True, bf16=False, optim=CONFIG["optim"], warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"], max_grad_norm=CONFIG["max_grad_norm"],
    logging_steps=10, logging_first_step=True, save_strategy="no", report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED, data_seed=SEED, dataset_num_proc=1,
)
rm_trainer = RewardTrainer(
    model=rm_model, args=rm_args, train_dataset=quality_rm_dataset,
    processing_class=tokenizer,
)
assert len(rm_trainer.train_dataset) == 2226, "No quality training pair may be filtered"
rm_result = rm_trainer.train()
assert rm_result.global_step > 0
rm_adapter_dir = runtime_root / "alignment_reward_model_adapter"
rm_model.save_pretrained(rm_adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(rm_adapter_dir)
print("Reward Model metrics:", rm_result.metrics)
print("Runtime RM adapter:", rm_adapter_dir)


In [ ]:
quality_public_rows, quality_public_hash = read_jsonl_with_sha256(quality_public_path)
assert quality_public_hash == CONFIG["public_test_quality_sha256"]
assert len(quality_public_rows) == 50
assert all(set(row) == {"prompt", "chosen", "rejected"} for row in quality_public_rows)
quality_train_triples = {
    (row["instruction"], row["chosen"], row["rejected"]) for row in quality_rows
}
quality_public_triples = {
    (row["prompt"], row["chosen"], row["rejected"]) for row in quality_public_rows
}
assert quality_public_triples.isdisjoint(quality_train_triples)
quality_public_full_lengths = [
    len(tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]},
         {"role": "assistant", "content": row[field]}],
        tokenize=True, add_generation_prompt=False,
    ))
    for row in quality_public_rows for field in ("chosen", "rejected")
]
assert max(quality_public_full_lengths) <= CONFIG["quality_max_length"]
print("Quality public max full tokens:", max(quality_public_full_lengths))
print("Task 3 public rows loaded after RM training:", len(quality_public_rows))

release_trainer(rm_trainer)
rm_model.gradient_checkpointing_disable()
rm_model.eval()

def reward_score(prompt: str, answer: str) -> float:
    encoded = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
        tokenize=True, add_generation_prompt=False, return_tensors="pt",
    ).to(rm_model.device)
    with torch.inference_mode():
        value = rm_model(
            input_ids=encoded, attention_mask=torch.ones_like(encoded), use_cache=False
        ).logits.squeeze().float().item()
    return value

rm_eval = []
for index, row in enumerate(quality_public_rows):
    chosen_reward = reward_score(row["prompt"], row["chosen"])
    rejected_reward = reward_score(row["prompt"], row["rejected"])
    rm_eval.append({
        "index": index, "chosen_reward": chosen_reward,
        "rejected_reward": rejected_reward,
        "margin": chosen_reward - rejected_reward,
        "correct": chosen_reward > rejected_reward,
    })
    print(f"Scored RM pair {index + 1:02d}/50")
rm_eval_frame = pd.DataFrame(rm_eval)
display(rm_eval_frame)
task3_metric = float(rm_eval_frame["correct"].mean())
task3_letter, task3_bounds = interval(task3_metric, [
    ("А", "0.6 <= accuracy < 0.75", lambda x: 0.6 <= x < 0.75),
    ("Б", "accuracy < 0.6", lambda x: x < 0.6),
    ("В", "0.9 <= accuracy <= 1.0", lambda x: 0.9 <= x <= 1.0),
    ("Г", "0.75 <= accuracy < 0.9", lambda x: 0.75 <= x < 0.9),
])
print("TASK3_NUM_PAIRS=50")
print(f"TASK3_SEED={SEED}")
print(f"TASK3_PAIRWISE_ACCURACY={task3_metric:.6f}")
print(f"TASK3_INTERVAL={task3_bounds}")
print(f"TASK3_ANSWER={task3_letter}")


## 6. Задача 4 — DPO по качеству

Reward Model выгружается. Новая policy и её reference загружаются из одного и того же style-DPO
адаптера, который был обучен выше в этой сессии. `chosen=good`, `rejected=bad`. Публичная метрика
не генерирует ответы и сравнивает средний logprob на токен ответа.


In [ ]:
del rm_trainer, rm_model
gc.collect()
torch.cuda.empty_cache()
model = load_style_checkpoint_trainable(style_adapter_dir)
model.load_adapter(style_adapter_dir, adapter_name="reference", is_trainable=False)
model.set_adapter("default")
policy_params = normalized_adapter_parameters(model, "default")
reference_params = normalized_adapter_parameters(model, "reference")
assert policy_params.keys() == reference_params.keys() and policy_params
assert all(torch.equal(policy_params[k].detach(), reference_params[k].detach()) for k in policy_params)
assert not any(p.requires_grad for p in reference_params.values())

quality_dpo_args = DPOConfig(
    output_dir=str(runtime_root / "alignment_quality_dpo_training"),
    num_train_epochs=CONFIG["quality_dpo_epochs"], learning_rate=CONFIG["quality_dpo_lr"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_prompt_length=CONFIG["quality_max_prompt_length"],
    max_length=CONFIG["quality_max_length"], beta=CONFIG["dpo_beta"],
    loss_type="sigmoid", model_adapter_name="default", ref_adapter_name="reference",
    fp16=True, bf16=False, optim=CONFIG["optim"], warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"], max_grad_norm=CONFIG["max_grad_norm"],
    logging_steps=10, logging_first_step=True, save_strategy="no", report_to="none",
    disable_dropout=True, precompute_ref_log_probs=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED, data_seed=SEED, dataset_num_proc=1,
)
quality_dpo_trainer = DPOTrainer(
    model=model, ref_model=None, args=quality_dpo_args,
    train_dataset=quality_pref_dataset, processing_class=tokenizer,
)
assert len(quality_dpo_trainer.train_dataset) == 2226
quality_dpo_result = quality_dpo_trainer.train()
assert quality_dpo_result.global_step > 0
quality_dpo_logs = [
    x for x in quality_dpo_trainer.state.log_history if "rewards/margins" in x
]
assert quality_dpo_logs
quality_dpo_adapter_dir = runtime_root / "alignment_quality_dpo_adapter"
model.set_adapter("default")
model.save_pretrained(
    quality_dpo_adapter_dir, safe_serialization=True, selected_adapters=["default"]
)
print("Quality-DPO metrics:", quality_dpo_result.metrics)
print("Reward margin first/last:", quality_dpo_logs[0]["rewards/margins"], quality_dpo_logs[-1]["rewards/margins"])


In [ ]:
release_trainer(quality_dpo_trainer)
model.set_adapter("default")
model.gradient_checkpointing_disable()
model.config.use_cache = False
model.eval()

def mean_answer_token_logprob(active_model, prompt: str, answer: str) -> tuple[float, float, int]:
    # The scored sequence is exactly: official chat generation prefix + raw answer tokens.
    # No prompt token and no chat terminator contributes to the numerator or denominator.
    prefix_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True,
        add_generation_prompt=True,
    )
    answer_ids = tokenizer(answer, add_special_tokens=False).input_ids
    assert prefix_ids and answer_ids
    input_ids = torch.tensor([prefix_ids + answer_ids], dtype=torch.long, device=active_model.device)
    with torch.inference_mode():
        logits = active_model(
            input_ids=input_ids, attention_mask=torch.ones_like(input_ids), use_cache=False
        ).logits
    # Position prefix_len-1 predicts answer token 0; the final input position predicts nothing scored.
    answer_logits = logits[0, len(prefix_ids) - 1:-1, :].float()
    targets = input_ids[0, len(prefix_ids):]
    assert answer_logits.shape[0] == targets.numel() == len(answer_ids)
    target_logits = answer_logits.gather(1, targets.unsqueeze(1)).squeeze(1)
    token_logprobs = target_logits - torch.logsumexp(answer_logits, dim=1)
    raw_sum = float(token_logprobs.sum().item())
    mean_logprob = raw_sum / len(answer_ids)
    del logits, answer_logits, target_logits, token_logprobs
    return mean_logprob, raw_sum, len(answer_ids)

def implicit_preference_evaluation(active_model, rows: list[dict], label: str):
    records = []
    for index, row in enumerate(rows):
        chosen_mean, chosen_sum, chosen_tokens = mean_answer_token_logprob(
            active_model, row["prompt"], row["chosen"]
        )
        rejected_mean, rejected_sum, rejected_tokens = mean_answer_token_logprob(
            active_model, row["prompt"], row["rejected"]
        )
        records.append({
            "index": index,
            "chosen_mean_logprob": chosen_mean,
            "rejected_mean_logprob": rejected_mean,
            "mean_margin": chosen_mean - rejected_mean,
            "chosen_raw_sum": chosen_sum,
            "rejected_raw_sum": rejected_sum,
            "chosen_tokens": chosen_tokens,
            "rejected_tokens": rejected_tokens,
            "correct": chosen_mean > rejected_mean,
        })
        print(f"{label}: scored pair {index + 1:02d}/{len(rows)}")
    frame = pd.DataFrame(records)
    assert len(frame) == 50 and frame["correct"].notna().all()
    return frame, float(frame["correct"].mean())

task4_frame, task4_metric = implicit_preference_evaluation(
    model, quality_public_rows, "Quality-DPO"
)
display(task4_frame)
task4_letter, task4_bounds = interval(task4_metric, [
    ("А", "0.6 <= accuracy < 0.75", lambda x: 0.6 <= x < 0.75),
    ("Б", "0.75 <= accuracy < 0.9", lambda x: 0.75 <= x < 0.9),
    ("В", "0.9 <= accuracy <= 1.0", lambda x: 0.9 <= x <= 1.0),
    ("Г", "accuracy < 0.6", lambda x: x < 0.6),
])
print("TASK4_NUM_PAIRS=50")
print(f"TASK4_SEED={SEED}")
print("TASK4_LENGTH_NORMALIZED=True")
print("TASK4_GENERATION=False")
print(f"TASK4_IMPLICIT_PREFERENCE_ACCURACY={task4_metric:.6f}")
print(f"TASK4_INTERVAL={task4_bounds}")
print(f"TASK4_ANSWER={task4_letter}")


## 7. Задача 5 — pure SimPO по качеству

Quality-DPO policy полностью выгружается. SimPO начинает с той же сохранённой style-DPO точки,
но не создаёт и не использует reference adapter/model. В TRL чистый SimPO задаётся
`loss_type="simpo"` и `cpo_alpha=0.0`; loss использует length-normalized logprob.


In [ ]:
del quality_dpo_trainer, model
gc.collect()
torch.cuda.empty_cache()
simpo_model = load_style_checkpoint_trainable(style_adapter_dir)
assert set(simpo_model.peft_config) == {"default"}

from trl import CPOConfig, CPOTrainer

simpo_args = CPOConfig(
    output_dir=str(runtime_root / "alignment_simpo_training"),
    num_train_epochs=CONFIG["simpo_epochs"], learning_rate=CONFIG["simpo_lr"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_prompt_length=CONFIG["quality_max_prompt_length"],
    max_length=CONFIG["quality_max_length"],
    loss_type="simpo", beta=CONFIG["simpo_beta"],
    simpo_gamma=CONFIG["simpo_gamma"], cpo_alpha=CONFIG["simpo_cpo_alpha"],
    fp16=True, bf16=False, optim=CONFIG["optim"], warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"], max_grad_norm=CONFIG["max_grad_norm"],
    logging_steps=10, logging_first_step=True, save_strategy="no", report_to="none",
    disable_dropout=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED, data_seed=SEED, dataset_num_proc=1,
)
assert simpo_args.loss_type == "simpo"
assert simpo_args.cpo_alpha == 0.0
assert simpo_args.simpo_gamma / simpo_args.beta == 0.5
simpo_trainer = CPOTrainer(
    model=simpo_model, args=simpo_args,
    train_dataset=quality_pref_dataset, processing_class=tokenizer,
)
assert len(simpo_trainer.train_dataset) == 2226
assert set(simpo_model.peft_config) == {"default"}, "SimPO must not create a reference adapter"
simpo_result = simpo_trainer.train()
assert simpo_result.global_step > 0
simpo_adapter_dir = runtime_root / "alignment_simpo_adapter"
simpo_model.save_pretrained(simpo_adapter_dir, safe_serialization=True)
print("SimPO metrics:", simpo_result.metrics)
print("Reference adapters present:", sorted(simpo_model.peft_config))


In [ ]:
release_trainer(simpo_trainer)
simpo_model.gradient_checkpointing_disable()
simpo_model.config.use_cache = False
simpo_model.eval()
task5_frame, task5_metric = implicit_preference_evaluation(
    simpo_model, quality_public_rows, "SimPO"
)
display(task5_frame)
task5_letter, task5_bounds = interval(task5_metric, [
    ("А", "0.9 <= accuracy <= 1.0", lambda x: 0.9 <= x <= 1.0),
    ("Б", "accuracy < 0.6", lambda x: x < 0.6),
    ("В", "0.6 <= accuracy < 0.75", lambda x: 0.6 <= x < 0.75),
    ("Г", "0.75 <= accuracy < 0.9", lambda x: 0.75 <= x < 0.9),
])
print("TASK5_NUM_PAIRS=50")
print(f"TASK5_SEED={SEED}")
print("TASK5_LENGTH_NORMALIZED=True")
print("TASK5_GENERATION=False")
print("TASK5_REFERENCE_MODEL=False")
print(f"TASK5_IMPLICIT_PREFERENCE_ACCURACY={task5_metric:.6f}")
print(f"TASK5_INTERVAL={task5_bounds}")
print(f"TASK5_ANSWER={task5_letter}")


## 8. Единый машинно-читаемый итог и проверки

Эта ячейка не содержит заранее вписанных результатов: она печатает пять значений, вычисленных выше
в текущем полном запуске.


In [ ]:
assert SEED == 42
assert len(task1_answers) == len(task2_answers) == 50
assert len(rm_eval_frame) == len(task4_frame) == len(task5_frame) == 50
assert CONFIG["simpo_cpo_alpha"] == 0.0
for metric in (task1_metric, task2_metric, task3_metric, task4_metric, task5_metric):
    assert np.isfinite(metric) and 0.0 <= metric <= 1.0

final_results = pd.DataFrame([
    {"task": 1, "metric": "mean P_simple", "value": task1_metric,
     "interval": task1_bounds, "answer": task1_letter},
    {"task": 2, "metric": "mean P_simple", "value": task2_metric,
     "interval": task2_bounds, "answer": task2_letter},
    {"task": 3, "metric": "RM pairwise accuracy", "value": task3_metric,
     "interval": task3_bounds, "answer": task3_letter},
    {"task": 4, "metric": "length-normalized implicit accuracy", "value": task4_metric,
     "interval": task4_bounds, "answer": task4_letter},
    {"task": 5, "metric": "length-normalized implicit accuracy", "value": task5_metric,
     "interval": task5_bounds, "answer": task5_letter},
])
display(final_results)
print("FINAL_SEED=42")
print(f"FINAL_TASK1_METRIC={task1_metric:.6f};ANSWER={task1_letter}")
print(f"FINAL_TASK2_METRIC={task2_metric:.6f};ANSWER={task2_letter}")
print(f"FINAL_TASK3_METRIC={task3_metric:.6f};ANSWER={task3_letter}")
print(f"FINAL_TASK4_METRIC={task4_metric:.6f};ANSWER={task4_letter}")
print(f"FINAL_TASK5_METRIC={task5_metric:.6f};ANSWER={task5_letter}")
print(f"DPO_VS_SIMPO_ACCURACY_DELTA={task5_metric - task4_metric:+.6f}")
